**TLDR**: Measures where the time goes in the selected pipeline (HOG, k = 50 PCA components, logistic regression) and compares baseline and improved implementations using the profiling, timing, FLOP-count, arithmetic-intensity and roofline methods from Workshop 2. Three optimisations are evaluated: reduced-resolution JPEG decoding for inference, randomized versus full SVD for PCA, and Newton's method versus gradient descent. Each is checked for both speed and accuracy. Helper functions are in `src/porkqc/perf.py` with 4 pytest tests, on the `feature/optimisation` branch.


## Method

1. Record the hardware, because Colab assigns different machines between sessions.
2. Time each stage of inference per image and profile feature extraction with `cProfile`.
3. Baseline vs improved:
   - JPEG decoding: full resolution vs decoder down-scaling (`Image.draft`), checked by grouped CV and validation AUC.
   - PCA: full thin SVD vs randomized SVD, checked by subspace agreement.
   - Classifier training: gradient descent vs Newton's method, across conditioning.
4. Estimate FLOPs and arithmetic intensity for each kernel and place them on a measured roofline.
5. Estimate memory requirements.


## Set-up


In [1]:
from google.colab import drive, userdata
drive.mount('/content/drive')

REPO_DIR = "/content/drive/MyDrive/MMA3001 Project/Pork-Tray-Inspection"
ZIP_PATH = "/content/drive/MyDrive/MMA3001/Pork Rasher Error-Packaging-.v4i.coco.zip"
%cd "{REPO_DIR}"

token = userdata.get("GITHUB_TOKEN")
PUSH_URL = f"https://{token}@github.com/chrixxv1/Pork-Tray-Inspection.git"

!mkdir -p /content/data
!unzip -oq "{ZIP_PATH}" -d /content/data
DATA_DIR = "/content/data"
CACHE_DIR = f"{REPO_DIR}/results/cache"

!pip install -q -e .
import sys
sys.path.insert(0, f"{REPO_DIR}/src")
!git branch


Mounted at /content/drive
/content/drive/MyDrive/MMA3001 Project/Pork-Tray-Inspection
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for porkqc (pyproject.toml) ... done
  feature/data-loader
  feature/experiment
  feature/features
  feature/logistic
  feature/pca
* main


## Branching


In [2]:
!git switch main
!git pull --no-rebase --no-edit {PUSH_URL} main
!git switch -c feature/optimisation


M	notebooks/6_experiment.ipynb
Already on 'main'
Your branch is ahead of 'origin/main' by 25 commits.
  (use "git push" to publish your local commits)
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 6 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 1.84 KiB | 1024 bytes/s, done.
From https://github.com/chrixxv1/Pork-Tray-Inspection
 * branch            main       -> FETCH_HEAD
Updating 25d98ca..f3fbd3f
Fast-forward
 AI_USE_LOG.md                    | 1 -
 notebooks/6_experiment (1).ipynb | 1 -
 2 files changed, 2 deletions(-)
 delete mode 100644 notebooks/6_experiment (1).ipynb
Switched to a new branch 'feature/optimisation'


## Performance Module

Timing, FLOP and arithmetic-intensity estimates, and reduced-resolution JPEG decoding.


In [3]:
%%writefile src/porkqc/perf.py
"""Performance helpers: timing, FLOP and arithmetic-intensity estimates, fast decoding.

FLOP counts are analytic estimates that count a multiply and an add as two
floating-point operations. Arithmetic intensity (AI) is FLOPs per byte moved
between memory and the processor, as in a roofline analysis: kernels with low AI
are limited by memory bandwidth, kernels with high AI by peak arithmetic rate.
All byte counts assume float64 (8 bytes) and that each array is read from
memory once per pass, i.e. no reuse from cache.
"""

from __future__ import annotations

import time
from pathlib import Path
from typing import Callable

import numpy as np
from PIL import Image

BYTES_PER_FLOAT = 8


def time_best(fn: Callable[[], object], repeats: int = 5) -> float:
    """Best (minimum) wall-clock time of ``fn()`` over ``repeats`` runs, in seconds.

    The minimum is used, as ``timeit`` does, because other processes can only
    make a run slower, never faster.

    Args:
        fn: Function with no arguments.
        repeats: Number of runs.

    Returns:
        The shortest run time in seconds.
    """
    if repeats < 1:
        raise ValueError("repeats must be at least 1")
    best = float("inf")
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn()
        best = min(best, time.perf_counter() - t0)
    return best


def gd_iteration(n: int, p: int) -> tuple[float, float]:
    """FLOPs and bytes of one gradient-descent iteration.

    Two matrix-vector products with the ``n x p`` matrix ``A`` (``A theta`` and
    ``A^T r``), each ``2 n p`` FLOPs, each reading ``A`` once.

    Args:
        n: Number of samples.
        p: Number of parameters (features + 1).

    Returns:
        ``(flops, bytes)``.
    """
    return 4.0 * n * p, 2.0 * n * p * BYTES_PER_FLOAT


def hessian_formation(n: int, p: int) -> tuple[float, float]:
    """FLOPs and bytes of forming ``H = A^T diag(s) A`` for an ``n x p`` matrix.

    Scaling the rows costs ``n p``; the product ``(A * s)^T A`` costs about
    ``2 n p^2``. ``A`` is read twice and the scaled copy written once.

    Args:
        n: Number of samples.
        p: Number of parameters.

    Returns:
        ``(flops, bytes)``.
    """
    return 2.0 * n * p**2 + n * p, 3.0 * n * p * BYTES_PER_FLOAT


def newton_iteration(n: int, p: int) -> tuple[float, float]:
    """FLOPs and bytes of one Newton iteration.

    Gradient (``4 n p``), Hessian formation (``2 n p^2 + n p``), LU
    factorisation (``2 p^3 / 3``) and the two triangular solves (``2 p^2``).

    Args:
        n: Number of samples.
        p: Number of parameters.

    Returns:
        ``(flops, bytes)``.
    """
    gf, gb = gd_iteration(n, p)
    hf, hb = hessian_formation(n, p)
    return gf + hf + 2.0 * p**3 / 3 + 2.0 * p**2, gb + hb


def randomized_svd_flops(n: int, d: int, r: int, q: int) -> float:
    """FLOPs of the randomized SVD of an ``n x d`` matrix with ``r`` columns and ``q`` power iterations.

    There are ``2q + 2`` products with the data matrix (``2 n d r`` each) plus
    Householder QR factorisations of tall ``n x r`` and ``d x r`` matrices
    (about ``4 m r^2`` each): ``q + 1`` of the first kind and ``q`` of the second.

    Args:
        n: Rows of the data matrix.
        d: Columns of the data matrix.
        r: Number of sampled directions (components + oversampling).
        q: Number of power iterations.

    Returns:
        Estimated FLOPs.
    """
    return 2.0 * n * d * r * (2 * q + 2) + 4.0 * ((q + 1) * n + q * d) * r**2


def full_svd_flops(n: int, d: int) -> float:
    """Lower-bound FLOPs of a thin SVD (singular values and right vectors).

    Uses ``4 m k^2 + 8 k^3`` with ``m = max(n, d)`` and ``k = min(n, d)``
    (Golub & Van Loan, 2013, ch. 8); forming the left vectors as well, as
    ``numpy.linalg.svd`` does, costs more.

    Args:
        n: Rows.
        d: Columns.

    Returns:
        Estimated FLOPs.
    """
    m, k = max(n, d), min(n, d)
    return 4.0 * m * k**2 + 8.0 * k**3


def arithmetic_intensity(flops: float, nbytes: float) -> float:
    """Arithmetic intensity in FLOP per byte.

    Args:
        flops: Floating-point operations.
        nbytes: Bytes moved.

    Returns:
        ``flops / nbytes``.
    """
    if nbytes <= 0:
        raise ValueError("nbytes must be positive")
    return flops / nbytes


def load_image_draft(path: str | Path, size: tuple[int, int] = (360, 270)) -> np.ndarray:
    """Load a JPEG at reduced resolution using the decoder's DCT scaling.

    ``Image.draft`` asks the JPEG decoder to skip detail it would only throw
    away: decoding at 1/2 scale (or 1/4, 1/8) is much cheaper than decoding at
    full size and resizing afterwards. HOG features are computed at 240 x 180,
    so decoding at 360 x 270 discards nothing that the features use.

    Args:
        path: Path to a JPEG file.
        size: Requested (width, height); the decoder picks the nearest scale
            that is at least this large.

    Returns:
        RGB array in [0, 1] of shape ``(H, W, 3)``.

    Raises:
        FileNotFoundError: If ``path`` does not exist.
    """
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(path)
    with Image.open(path) as im:
        im.draft("RGB", size)
        return np.asarray(im.convert("RGB"), dtype=np.float64) / 255.0



Writing src/porkqc/perf.py


## Tests for the Performance Module


In [4]:
%%writefile tests/test_perf.py
"""Tests of the performance helpers."""

import numpy as np
import pytest
from PIL import Image

from porkqc.features import hog_features
from porkqc.perf import (arithmetic_intensity, gd_iteration, hessian_formation, load_image_draft,
                         newton_iteration, randomized_svd_flops, time_best)


def test_flop_counts_match_hand_calculation():
    # n = 10, p = 3: GD = 4*10*3 = 120 FLOPs, reading A twice = 2*10*3*8 = 480 bytes.
    assert gd_iteration(10, 3) == (120.0, 480.0)
    # Hessian: 2*10*9 + 30 = 210 FLOPs.
    assert hessian_formation(10, 3)[0] == 210.0
    # Newton adds LU (2*27/3 = 18) and two solves (2*9 = 18) to GD + Hessian.
    assert newton_iteration(10, 3)[0] == 120 + 210 + 18 + 18
    # Randomized SVD, q = 0: two passes of 2*n*d*r plus one n x r QR.
    assert randomized_svd_flops(4, 5, 2, 0) == 2 * 4 * 5 * 2 * 2 + 4 * 4 * 4


def test_gd_is_memory_bound_and_hessian_intensity_grows_with_p():
    assert np.isclose(arithmetic_intensity(*gd_iteration(1000, 51)), 0.25)
    small = arithmetic_intensity(*hessian_formation(1000, 11))
    large = arithmetic_intensity(*hessian_formation(1000, 401))
    assert large > 10 * small
    with pytest.raises(ValueError):
        arithmetic_intensity(1.0, 0.0)


def test_time_best_is_positive_and_validates():
    assert time_best(lambda: sum(range(1000)), repeats=3) > 0
    with pytest.raises(ValueError):
        time_best(lambda: None, repeats=0)


def test_draft_decoding_halves_resolution_and_keeps_feature_length(tmp_path):
    rng = np.random.default_rng(0)
    path = tmp_path / "frame.jpg"
    Image.fromarray(rng.integers(0, 255, (540, 720, 3), dtype=np.uint8)).save(path)
    rgb = load_image_draft(path)
    assert rgb.shape == (270, 360, 3)
    assert 0.0 <= rgb.min() and rgb.max() <= 1.0
    assert hog_features(rgb).shape == (5040,)
    with pytest.raises(FileNotFoundError):
        load_image_draft(tmp_path / "missing.jpg")



Writing tests/test_perf.py


## Run Tests
Expected: `57 passed`.


In [5]:
!pytest -q


.........................................................                [100%]
57 passed in 10.16s


## Hardware

Colab gives a different machine between sessions, so all timings below belong to the hardware printed here. Record it in the report.


In [6]:
!lscpu | grep -E "Model name|^CPU\(s\)|Thread\(s\) per core|L3 cache"
import numpy as np
print("NumPy", np.__version__)


CPU(s):                                  2
Model name:                              Intel(R) Xeon(R) CPU @ 2.20GHz
Thread(s) per core:                      2
L3 cache:                                55 MiB (1 instance)
NumPy 2.1.3


## Load Features and Fit the Selected Model


In [7]:
import json, time, cProfile, pstats, io, os, warnings
import matplotlib.pyplot as plt
from porkqc.data import load_dataset
from porkqc.features import extract, hog_features, load_image
from porkqc.pca import PCA
from porkqc.logistic import LogisticRegression
from porkqc.evaluate import roc_auc
from porkqc.selection import frame_block_groups
from porkqc import perf

K, L2 = 50, 1e-4   # selected in Phase 6
ds = load_dataset(DATA_DIR)
(X_tr, y_tr), (X_va, y_va), (X_te, y_te) = (extract(ds[s], "hog", cache_dir=CACHE_DIR)
                                            for s in ("train", "valid", "test"))
pca = PCA(K).fit(X_tr)
model = LogisticRegression(l2=L2, tol=1e-10).fit(pca.transform(X_tr), y_tr)
n, d = X_tr.shape
os.makedirs("results/figures", exist_ok=True)
R = {"n_train": n, "n_features": d, "k": K}
print(f"Training matrix {n} x {d}; validation AUC {roc_auc(y_va, model.predict_proba(pca.transform(X_va))):.3f}")


Training matrix 3349 x 5040; validation AUC 0.784


## Per-Image Inference Latency

The stages a production-line system would run for each tray: decode the JPEG, compute HOG, then project onto the PCA components and classify. The median over 50 test images is reported.


In [8]:
def latency(loader, records):
    t_load, t_hog, t_cls = [], [], []
    for r in records:
        t0 = time.perf_counter(); rgb = loader(r.path)
        t1 = time.perf_counter(); f = hog_features(rgb)
        t2 = time.perf_counter(); model.predict_proba(pca.transform(f[None]))
        t3 = time.perf_counter()
        t_load.append(t1 - t0); t_hog.append(t2 - t1); t_cls.append(t3 - t2)
    ms = {"decode_ms": 1e3 * np.median(t_load), "hog_ms": 1e3 * np.median(t_hog),
          "pca_and_classify_ms": 1e3 * np.median(t_cls)}
    ms["total_ms"] = sum(ms.values())
    ms["images_per_s"] = 1e3 / ms["total_ms"]
    return ms

records = ds["test"][:50]
R["latency_standard"] = latency(load_image, records)
for key, v in R["latency_standard"].items():
    print(f"{key:22s} {v:8.2f}")
# Inference FLOPs: centre (d), project (2dk), classify (2k).
R["predict_flops"] = d + 2 * d * K + 2 * K
print(f"\nPCA + classify: {R['predict_flops']:.2e} FLOPs per image")


decode_ms                 12.22
hog_ms                    18.05
pca_and_classify_ms        0.32
total_ms                  30.59
images_per_s              32.69

PCA + classify: 5.09e+05 FLOPs per image


## Profile Feature Extraction

`cProfile` shows which functions the time is spent in. Sorting by cumulative time shows the call chain; the `tottime` column shows the time spent inside each function itself.


In [9]:
prof = cProfile.Profile()
prof.enable()
for r in records[:20]:
    hog_features(load_image(r.path))
prof.disable()
s = io.StringIO()
pstats.Stats(prof, stream=s).sort_stats("cumulative").print_stats(12)
print(s.getvalue()[:3500])
with open("results/7_profile_feature_extraction.txt", "w") as f:
    f.write(s.getvalue())


         46934 function calls (46928 primitive calls) in 0.989 seconds

   Ordered by: cumulative time
   List reduced from 240 to 12 due to restriction <12>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
      3/2    0.000    0.000    0.986    0.493 /usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3512(run_code)
      3/2    0.000    0.000    0.986    0.493 {built-in method builtins.exec}
        1    0.001    0.001    0.897    0.897 /tmp/ipykernel_2190/1118603354.py:1(<cell line: 0>)
       20    0.006    0.000    0.602    0.030 /content/drive/MyDrive/MMA3001 Project/Pork-Tray-Inspection/src/porkqc/features.py:83(hog_features)
       20    0.114    0.006    0.358    0.018 /content/drive/MyDrive/MMA3001 Project/Pork-Tray-Inspection/src/porkqc/features.py:35(load_image)
       20    0.012    0.001    0.338    0.017 /usr/local/lib/python3.13/dist-packages/skimage/_shared/utils.py:440(fixed_func)
       20    0.162    0.008    0.325    0.

## Optimisation 1: Reduced-Resolution JPEG Decoding

The profile shows that most of the time goes into decoding the full 720 × 540 image and resizing it, even though HOG only uses a 240 × 180 version. `Image.draft` asks the JPEG decoder to decode at half resolution directly, which is much cheaper. Because this changes the features slightly, accuracy is checked the same way the model was selected: grouped 5-fold CV on the training split, plus the validation AUC.


In [10]:
R["latency_draft"] = latency(perf.load_image_draft, records)
print(f"{'':22s}{'standard':>10s}{'draft':>10s}")
for key in R["latency_standard"]:
    print(f"{key:22s}{R['latency_standard'][key]:10.2f}{R['latency_draft'][key]:10.2f}")
speedup = R["latency_standard"]["total_ms"] / R["latency_draft"]["total_ms"]
R["draft_speedup"] = speedup
print(f"\nSpeed-up per image: {speedup:.1f}x")


                        standard     draft
decode_ms                  12.22      4.42
hog_ms                     18.05     21.97
pca_and_classify_ms         0.32      0.41
total_ms                   30.59     26.81
images_per_s               32.69     37.31

Speed-up per image: 1.1x


In [11]:
# Accuracy check: re-extract features with draft decoding (about 1 minute), then
# grouped CV and validation AUC with the selected settings.
t0 = time.perf_counter()
X_draft = {s: np.stack([hog_features(perf.load_image_draft(r.path)) for r in ds[s]])
           for s in ("train", "valid")}
print(f"Draft feature extraction for train + valid: {time.perf_counter() - t0:.0f} s")

from sklearn.model_selection import GroupKFold
groups = frame_block_groups(ds["train"], block=100)

def cv_and_valid(Xtr, Xva):
    aucs = []
    for tr, te in GroupKFold(5).split(Xtr, y_tr, groups):
        p = PCA(K, solver="randomized").fit(Xtr[tr])
        c = LogisticRegression(l2=L2).fit(p.transform(Xtr[tr]), y_tr[tr])
        aucs.append(roc_auc(y_tr[te], c.predict_proba(p.transform(Xtr[te]))))
    p = PCA(K).fit(Xtr)
    c = LogisticRegression(l2=L2).fit(p.transform(Xtr), y_tr)
    return float(np.mean(aucs)), float(np.std(aucs) / np.sqrt(5)), roc_auc(y_va, c.predict_proba(p.transform(Xva)))

rel = np.linalg.norm(X_draft["valid"] - X_va) / np.linalg.norm(X_va)
R["draft_accuracy"] = {"standard": cv_and_valid(X_tr, X_va), "draft": cv_and_valid(X_draft["train"], X_draft["valid"]),
                       "relative_feature_change": float(rel)}
print(f"Relative change in HOG features: {rel:.2f}")
for name in ("standard", "draft"):
    cv, se, va = R["draft_accuracy"][name]
    print(f"{name:9s} grouped CV AUC {cv:.3f} ± {se:.3f}, validation AUC {va:.3f}")


Draft feature extraction for train + valid: 74 s
Relative change in HOG features: 0.17
standard  grouped CV AUC 0.722 ± 0.009, validation AUC 0.784
draft     grouped CV AUC 0.719 ± 0.011, validation AUC 0.784


## Measure This Machine's Roofline

A roofline needs the machine's peak arithmetic rate and its memory bandwidth. The peak is estimated with a large matrix multiplication (2N³ FLOPs, compute-bound). Bandwidth is estimated with a vector addition `a = b + c` (one FLOP per 24 bytes moved, memory-bound), measured twice: with arrays far larger than the CPU cache (main-memory bandwidth) and with arrays small enough to stay in cache (cache bandwidth). This is the "perfectly cached vs reloaded variables" distinction from Workshop 2.


In [12]:
N = 2000
A1, B1 = np.random.rand(N, N), np.random.rand(N, N)
t = perf.time_best(lambda: A1 @ B1, repeats=3)
peak_gflops = 2 * N**3 / t / 1e9
del A1, B1

def add_bandwidth(m, repeats):
    b, c, a = np.random.rand(m), np.random.rand(m), np.empty(m)
    t = perf.time_best(lambda: np.add(b, c, out=a), repeats=repeats)
    return 24 * m / t / 1e9

bandwidth_gbs = add_bandwidth(20_000_000, 5)     # 480 MB of arrays: main memory
cache_bandwidth_gbs = add_bandwidth(50_000, 200)  # 1.2 MB of arrays: stays in cache

R["peak_gflops"], R["bandwidth_gbs"], R["cache_bandwidth_gbs"] = peak_gflops, bandwidth_gbs, cache_bandwidth_gbs
print(f"Peak (matmul): {peak_gflops:.1f} GFLOP/s")
print(f"Main-memory bandwidth: {bandwidth_gbs:.1f} GB/s, cache bandwidth: {cache_bandwidth_gbs:.1f} GB/s")
print(f"Ridge point (main memory): {peak_gflops / bandwidth_gbs:.1f} FLOP/byte; "
      "kernels below this arithmetic intensity are memory-bound")


Peak (matmul): 35.1 GFLOP/s
Main-memory bandwidth: 7.0 GB/s, cache bandwidth: 28.6 GB/s
Ridge point (main memory): 5.0 FLOP/byte; kernels below this arithmetic intensity are memory-bound


## Optimisation 2: Full vs Randomized SVD for PCA

The full thin SVD computes all min(n, d) = 3,349 components although only 50 are kept. The randomized SVD computes only the leading ones. Its accuracy depends on the number of power iterations q, so q = 0, 4 and 10 are compared with the full SVD: the singular values, and the cosines of the principal angles between the two 50-dimensional subspaces (1 means identical).


In [13]:
pca_rows = {}
t = perf.time_best(lambda: PCA(K, solver="full").fit(X_tr), repeats=1)
full = PCA(K + 1, solver="full").fit(X_tr)
flops = perf.full_svd_flops(n, d)
pca_rows["full"] = {"time_s": t, "gflops_est": flops / 1e9, "achieved_gflop_s": flops / t / 1e9}
gap = (full.singular_values_[K - 1] - full.singular_values_[K]) / full.singular_values_[K - 1]
R["spectral_gap_rel"] = float(gap)

for q in (0, 4, 10):
    t = perf.time_best(lambda: PCA(K, solver="randomized", n_power_iter=q).fit(X_tr), repeats=1)
    rnd = PCA(K, solver="randomized", n_power_iter=q).fit(X_tr)
    cos = np.linalg.svd(full.components_[:K] @ rnd.components_.T, compute_uv=False)
    flops = perf.randomized_svd_flops(n, d, K + 10, q)
    pca_rows[f"randomized_q{q}"] = {
        "time_s": t, "gflops_est": flops / 1e9, "achieved_gflop_s": flops / t / 1e9,
        "max_rel_sv_error": float(np.max(np.abs(rnd.singular_values_ - full.singular_values_[:K])) / full.singular_values_[0]),
        "min_cosine": float(cos.min()), "directions_cos_gt_0.99": int((cos > 0.99).sum())}
R["pca"] = pca_rows

print(f"Relative gap between singular values 50 and 51: {gap:.3f}\n")
print(f"{'solver':16s}{'time (s)':>10s}{'GFLOP est':>11s}{'GFLOP/s':>9s}{'min cos':>9s}{'dirs > 0.99':>12s}")
for name, r in pca_rows.items():
    print(f"{name:16s}{r['time_s']:10.2f}{r['gflops_est']:11.1f}{r['achieved_gflop_s']:9.1f}"
          f"{r.get('min_cosine', 1.0):9.3f}{r.get('directions_cos_gt_0.99', K):12d}")


Relative gap between singular values 50 and 51: 0.011

solver            time (s)  GFLOP est  GFLOP/s  min cos dirs > 0.99
full                 82.01      526.6      6.4    1.000          50
randomized_q0         1.36        4.1      3.0    0.026           7
randomized_q4         2.16       20.8      9.6    0.366          42
randomized_q10        3.25       45.8     14.1    0.989          49


## Optimisation 3: Newton's Method vs Gradient Descent

Both solvers are run to the same residual tolerance as the number of components k (and so the Hessian's condition number) grows.


In [ ]:
pca400 = PCA(400, solver="randomized").fit(X_tr)
Z400 = pca400.transform(X_tr)
solver_rows = []
for k in (10, 50, 200, 400):
    p = k + 1
    for solver, kw in (("newton", {}), ("gd", {"max_iter": 20_000})):
        t = perf.time_best(lambda: LogisticRegression(l2=L2, solver=solver, tol=1e-6, **kw).fit(Z400[:, :k], y_tr), repeats=1)
        m = LogisticRegression(l2=L2, solver=solver, tol=1e-6, **kw).fit(Z400[:, :k], y_tr)
        iters = len(m.history_.loss) - 1
        f_it, b_it = perf.newton_iteration(n, p) if solver == "newton" else perf.gd_iteration(n, p)
        solver_rows.append({"k": k, "solver": solver, "iterations": iters, "time_s": t,
                            "flops": f_it * iters, "ai": perf.arithmetic_intensity(f_it, b_it)})
    solver_rows[-1]["hessian_cond"] = solver_rows[-2]["hessian_cond"] = \
        LogisticRegression(l2=L2, tol=1e-6).fit(Z400[:, :k], y_tr).hessian_cond_
R["solvers"] = solver_rows

print(f"{'k':>4s}{'solver':>8s}{'iters':>7s}{'time (ms)':>11s}{'MFLOP':>9s}{'AI':>7s}{'cond(H)':>9s}")
for r in solver_rows:
    print(f"{r['k']:4d}{r['solver']:>8s}{r['iterations']:7d}{1e3 * r['time_s']:11.1f}"
          f"{r['flops'] / 1e6:9.1f}{r['ai']:7.2f}{r['hessian_cond']:9.1f}")


## Roofline Plot

Each kernel is timed on its own and placed at its arithmetic intensity. Two roofs are drawn: min(peak, AI × bandwidth) for data streamed from main memory, and the same with the cache bandwidth. A kernel whose data fits in cache (the 3,349 × 51 matrix used by the solvers is only 1.4 MB) can exceed the main-memory roof; the 135 MB HOG matrix cannot.


In [ ]:
p50 = K + 1
A50 = np.hstack([pca.transform(X_tr), np.ones((n, 1))])
theta = np.r_[model.coef_, model.intercept_]
sv = np.full(n, 0.25)
Omega = np.random.rand(d, K + 10)
x1 = X_te[:1]

kernels = {}
def add_kernel(name, fn, flops, nbytes, working_set, repeats=5):
    t = perf.time_best(fn, repeats)
    kernels[name] = {"ai": flops / nbytes, "gflop_s": flops / t / 1e9, "time_s": t,
                     "working_set_mb": working_set / 1e6}

add_kernel("GD iteration (2 mat-vec)", lambda: A50.T @ (A50 @ theta - y_tr),
           *perf.gd_iteration(n, p50), A50.nbytes)
add_kernel("Hessian A^T diag(s) A", lambda: (A50 * sv[:, None]).T @ A50,
           *perf.hessian_formation(n, p50), A50.nbytes)
add_kernel("Randomized-SVD pass X Ω", lambda: X_tr @ Omega,
           2.0 * n * d * (K + 10), 8.0 * (n * d + d * (K + 10)), X_tr.nbytes, repeats=3)
add_kernel("Inference projection", lambda: (x1 - pca.mean_) @ pca.components_.T,
           d + 2.0 * d * K, 8.0 * (d * K + 2 * d), pca.components_.nbytes)
R["roofline"] = {"peak_gflops": peak_gflops, "bandwidth_gbs": bandwidth_gbs,
                 "cache_bandwidth_gbs": cache_bandwidth_gbs, "kernels": kernels}

ai = np.logspace(-2, 3, 200)
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.loglog(ai, np.minimum(peak_gflops, ai * bandwidth_gbs), "k-", lw=2, label="roof: main memory")
ax.loglog(ai, np.minimum(peak_gflops, ai * cache_bandwidth_gbs), "k--", lw=1, label="roof: cache")
for name, kv in kernels.items():
    ax.loglog(kv["ai"], kv["gflop_s"], "o", ms=8,
              label=f"{name} ({kv['working_set_mb']:.1f} MB): {kv['gflop_s']:.1f} GFLOP/s")
ax.set(xlabel="arithmetic intensity (FLOP/byte)", ylabel="performance (GFLOP/s)",
       title="Roofline: where each kernel of the pipeline sits")
ax.grid(alpha=0.3, which="both")
ax.legend(fontsize=7, loc="lower right")
plt.tight_layout()
fig.savefig("results/figures/7_roofline.png", dpi=150, bbox_inches="tight")
plt.show()
for name, kv in kernels.items():
    print(f"{name:28s} AI {kv['ai']:6.2f} FLOP/byte, working set {kv['working_set_mb']:7.1f} MB, "
          f"{kv['gflop_s']:6.1f} GFLOP/s")


## Memory Requirements

Each float64 value takes 8 bytes (float32: 4 bytes).


In [ ]:
mem = {
    "HOG training matrix (n x d)": n * d * 8,
    "PCA components (k x d)": K * d * 8,
    "PCA mean (d)": d * 8,
    "Classifier weights (k + 1)": (K + 1) * 8,
    "One 720 x 540 RGB image as float64": 720 * 540 * 3 * 8,
}
R["memory_bytes"] = mem
for name, b in mem.items():
    print(f"{name:36s} {b / 1e6:10.3f} MB   (float32: {b / 2e6:.3f} MB)")
deploy = mem["PCA components (k x d)"] + mem["PCA mean (d)"] + mem["Classifier weights (k + 1)"]
print(f"\nDeployed model (PCA + weights): {deploy / 1e6:.2f} MB")


## Save Results


In [ ]:
with open("results/7_profile.json", "w") as f:
    json.dump(R, f, indent=2, default=float)
print("Saved results/7_profile.json")


## Documentation

This cell writes `docs/7_optimisation.md` from the measured values, so the summary always matches the hardware this notebook ran on.


In [ ]:
ls, ld = R["latency_standard"], R["latency_draft"]
(cv_s, se_s, va_s), (cv_d, se_d, va_d) = R["draft_accuracy"]["standard"], R["draft_accuracy"]["draft"]
pr = R["pca"]
solver_table = "\n".join(f"| {r['k']} | {r['solver']} | {r['iterations']} | {1e3 * r['time_s']:.1f} | "
                         f"{r['flops'] / 1e6:.1f} | {r['ai']:.2f} | {r['hessian_cond']:.1f} |" for r in R["solvers"])
pca_table = "\n".join(f"| {name} | {r['time_s']:.2f} | {r['gflops_est']:.1f} | {r['achieved_gflop_s']:.1f} | "
                       f"{r.get('min_cosine', 1.0):.3f} | {r.get('directions_cos_gt_0.99', K)} |" for name, r in pr.items())
kern_table = "\n".join(f"| {name} | {kv['ai']:.2f} | {kv['working_set_mb']:.1f} | {kv['gflop_s']:.1f} |" for name, kv in R["roofline"]["kernels"].items())

doc = f"""# Phase 7: Optimisation and performance

Notebook: `notebooks/7_optimisation.ipynb`. Code: `src/porkqc/perf.py`. Full results: `results/7_profile.json`.
Timings are for the Colab machine recorded in the notebook and vary between sessions.

## Machine

Measured peak {R['peak_gflops']:.1f} GFLOP/s (2000 x 2000 matrix multiplication), main-memory bandwidth
{R['bandwidth_gbs']:.1f} GB/s and cache bandwidth {R['cache_bandwidth_gbs']:.1f} GB/s (a = b + c).
Ridge point {R['peak_gflops'] / R['bandwidth_gbs']:.1f} FLOP/byte.

## Inference latency per image

| Stage | Standard decoding (ms) | Draft decoding (ms) |
|---|---|---|
| JPEG decode | {ls['decode_ms']:.2f} | {ld['decode_ms']:.2f} |
| HOG | {ls['hog_ms']:.2f} | {ld['hog_ms']:.2f} |
| PCA projection + classification | {ls['pca_and_classify_ms']:.2f} | {ld['pca_and_classify_ms']:.2f} |
| **Total** | **{ls['total_ms']:.2f}** | **{ld['total_ms']:.2f}** |
| Images per second | {ls['images_per_s']:.0f} | {ld['images_per_s']:.0f} |

Projection and classification take about {R['predict_flops'] / 1e6:.2f} MFLOP per image; decoding and HOG dominate.

## Optimisation 1: reduced-resolution JPEG decoding

Decoding at half resolution with `Image.draft` is {R['draft_speedup']:.1f}x faster per image. It changes the HOG
features by {100 * R['draft_accuracy']['relative_feature_change']:.0f} % (relative norm), but accuracy is unchanged within
one standard error: grouped CV AUC {cv_d:.3f} ± {se_d:.3f} vs {cv_s:.3f} ± {se_s:.3f}, validation AUC {va_d:.3f} vs {va_s:.3f}.

## Optimisation 2: full vs randomized SVD

Relative gap between singular values 50 and 51: {R['spectral_gap_rel']:.3f}.

| Solver | Time (s) | Estimated GFLOP | Achieved GFLOP/s | Min subspace cosine | Directions with cosine > 0.99 |
|---|---|---|---|---|---|
{pca_table}

The spectrum decays slowly at k = 50, so the randomized SVD needs about 10 power iterations to reproduce
the full SVD's subspace; with q = 10 it does so at a fraction of the cost.

## Optimisation 3: Newton's method vs gradient descent (tolerance 1e-6)

| k | Solver | Iterations | Time (ms) | MFLOP | AI (FLOP/byte) | cond(H) |
|---|---|---|---|---|---|---|
{solver_table}

Gradient descent is memory-bound (AI 0.25) and its iteration count grows with the condition number;
Newton's method does more work per iteration, at higher arithmetic intensity, but needs only a few.

## Roofline

| Kernel | AI (FLOP/byte) | Working set (MB) | Achieved GFLOP/s |
|---|---|---|---|
{kern_table}

The gradient-descent iteration and the inference projection have low arithmetic intensity (about 0.25
FLOP/byte), so they are memory-bound; they exceed the main-memory roof only because their data fits in
cache. The Hessian formation and the randomized-SVD pass have much higher intensity and run closer to peak.

{'!'}[Roofline](../results/figures/7_roofline.png)

## Memory

Training matrix {R['memory_bytes']['HOG training matrix (n x d)'] / 1e6:.0f} MB in float64; the deployed model
(PCA components, mean and weights) is {(R['memory_bytes']['PCA components (k x d)'] + R['memory_bytes']['PCA mean (d)'] + R['memory_bytes']['Classifier weights (k + 1)']) / 1e6:.2f} MB.

## Final design

- Inference: draft JPEG decoding + HOG + PCA (k = 50) + logistic regression, chosen because it is
  {R['draft_speedup']:.1f}x faster with no measurable loss of accuracy.
- PCA: randomized SVD with q = 10 for repeated fitting (cross-validation); the full SVD for the final model,
  where its cost is paid once.
- Classifier: Newton's method, which converges in a few iterations regardless of conditioning.
"""
with open("docs/7_optimisation.md", "w") as f:
    f.write(doc)
print(doc[:1500])


## AI Usage Log


In [ ]:
%%writefile -a AI_USE_LOG.md
| 2026-10-27 | Claude | Wrote the code and src/porkqc/perf.py: latency breakdown, cProfile profiling, reduced-resolution JPEG decoding, full vs randomized SVD, Newton vs gradient descent, FLOP and arithmetic-intensity estimates, measured roofline and memory estimates, following the Workshop 2 methods; 4 tests | Ran the notebook on Colab and checked the measurements and accuracy comparisons; ran all tests (57 passed); asked Claude to explain parts I did not understand |


## Commit and Push the Branch

In [ ]:
!git add src/porkqc/perf.py tests/test_perf.py notebooks/7_optimisation.ipynb docs/7_optimisation.md results/7_profile.json results/7_profile_feature_extraction.txt results/figures/7_roofline.png AI_USE_LOG.md
!git commit -m "Add performance analysis: profiling, draft JPEG decoding, randomized SVD, solver comparison and roofline"
!git push {PUSH_URL} feature/optimisation


## Update Main After Merging